## Install (Colab Only)

In [ ]:
# install
!pip install pyepo
!pip install mpax

Cloning into 'PyEPO'...
remote: Enumerating objects: 147, done.
remote: Counting objects: 100% (147/147), done.
remote: Compressing objects: 100% (133/133), done.
remote: Total 147 (delta 30), reused 64 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (147/147), 6.94 MiB | 5.08 MiB/s, done.
Resolving deltas: 100% (30/30), done.
Processing ./PyEPO/pkg
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.0/131.0 kB 8.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 106.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1

Remove the problematic hook from Google Colab.

In [ ]:
import sys
sys.meta_path = [hook for hook in sys.meta_path if not any(keyword in str(hook) for keyword in ["google.colab"])]

# Solving on MPAX with PDHG

[`MPAX`](https://github.com/MIT-Lu-Lab/MPAX) (Mathematical Programming in JAX) is a hardware-accelerated, batchable, differentiable LP solver based on the PDHG (Primal-Dual Hybrid Gradient) first-order method. Where Gurobi solves one LP at a time on the CPU, MPAX uses `jax.vmap` to solve a **batch** of LPs in one fused JIT call on CPU or GPU.

This notebook does two things:

1. Benchmark MPAX's batched LP solve against a Gurobi loop on the same knapsack LP relaxation.
2. Train several end-to-end predict-then-optimize losses (SPO+, PFYL, RFWO, RFYL, pointwise LTR) on top of the MPAX-backed model.

## 1 Setup

In [ ]:
import time
import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

import pyepo
from pyepo.model.grb import knapsackModel as grbKnapsack
from pyepo.model.mpax import knapsackModel as mpaxKnapsack

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# knapsack problem dimensions
m = 10000          # number of items
k = 50             # resource dimensions
caps = [500] * k   # per-resource capacity
p = 5              # feature dimension
deg = 4            # polynomial degree of the feature-to-cost mapping
noise = 0.5        # multiplicative noise half-width
n_train, n_test = 1000, 1000

weights, feats, costs = pyepo.data.knapsack.genData(
    num_data=n_train + n_test, num_features=p, num_items=m,
    dim=k, deg=deg, noise_width=noise, seed=42,
)
print(f'weights: {weights.shape}, feats: {feats.shape}, costs: {costs.shape}')

# both backends solve the LP relaxation of the same problem
mpax_optmodel = mpaxKnapsack(weights, caps)
grb_optmodel = grbKnapsack(weights, caps).relax()

## 2 Solver Speed: MPAX vs Gurobi

Both backends solve the same knapsack LP relaxation. Gurobi calls its sequential simplex / barrier code one instance at a time; MPAX issues a single `jax.vmap` call that batches all instances into one JIT-compiled solve. We time both on the same 20-instance subset of the dataset (kept small only because the Gurobi loop is the bottleneck for the demo; MPAX would happily batch thousands at once).

In [ ]:
N_bench = 20
bench_costs = costs[:N_bench]

# Gurobi: sequential loop
t0 = time.time()
for c in tqdm(bench_costs, desc='Gurobi'):
    grb_optmodel.setObj(c)
    grb_optmodel.solve()
grb_time = time.time() - t0

# MPAX: vmap batch (warm-up first so JIT compile is not charged)
mpax_optmodel.setObj(bench_costs[:1])
_ = mpax_optmodel.batch_optimize(mpax_optmodel.c)

t0 = time.time()
mpax_optmodel.setObj(bench_costs)
sols, objs = mpax_optmodel.batch_optimize(mpax_optmodel.c)
mpax_time = time.time() - t0

print(f'Gurobi (sequential): {grb_time:.2f}s total, {grb_time / N_bench * 1000:.1f} ms/instance')
print(f'MPAX   (vmap batch): {mpax_time:.2f}s total, {mpax_time / N_bench * 1000:.1f} ms/instance')
print(f'Speedup: {grb_time / mpax_time:.1f}x')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
labels = ['Gurobi\n(sequential loop)', 'MPAX\n(vmap batch)']
values = [grb_time, mpax_time]
colors = ['#ff9248', '#48a9ff']
bars = ax.bar(labels, values, color=colors)
ax.set_ylabel('Wall-clock time (s)')
ax.set_title(f'Solve {N_bench} knapsack LP instances (m={m} items, k={k} resources)')
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f'{v:.2f}s', ha='center', va='bottom')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 3 Dataset and Data Loader

Now we build the full training and test `optDataset` on the MPAX backend. For MPAX models, `optDataset` automatically takes a `vmap` fast path so the entire dataset is solved in a single dispatch instead of one solve per instance.

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, c_train, c_test = train_test_split(
    feats, costs, test_size=n_test, random_state=42,
)

In [ ]:
dataset_train = pyepo.data.dataset.optDataset(mpax_optmodel, x_train, c_train)
dataset_test = pyepo.data.dataset.optDataset(mpax_optmodel, x_test, c_test)

In [ ]:
batch_size = 32
loader_train = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
loader_test = DataLoader(dataset_test, batch_size=batch_size, shuffle=False)

## 4 Linear Predictor

A single `nn.Linear(p, m)` mapping features to per-item values. The optimization model is fixed -- only the cost vector is predicted from data.

In [ ]:
class LinearRegression(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(p, m)

    def forward(self, x):
        return self.linear(x)

## 5 Training

We train five end-to-end losses on top of the same MPAX-backed knapsack model. The training loop is shared; only the loss function changes.

In [ ]:
def trainModel(reg, loss_func, method_name, num_epochs=10, lr=1e-2):
    optimizer = torch.optim.Adam(reg.parameters(), lr=lr)
    reg.train()
    loss_log = []
    loss_log_regret = [pyepo.metric.regret(reg, mpax_optmodel, loader_test)]
    elapsed = 0.0
    for epoch in range(num_epochs):
        tick = time.time()
        for x, c, w, z in loader_train:
            if torch.cuda.is_available():
                x, c, w, z = x.cuda(), c.cuda(), w.cuda(), z.cuda()
            cp = reg(x)
            if method_name == 'spo+':
                loss = loss_func(cp, c, w, z)
            elif method_name in ['pfy', 'rfwo', 'rfyl']:
                loss = loss_func(cp, w)
            elif method_name == 'ltr':
                loss = loss_func(cp, c)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            elapsed += time.time() - tick
            tick = time.time()
            loss_log.append(loss.item())
        regret = pyepo.metric.regret(reg, mpax_optmodel, loader_test)
        loss_log_regret.append(regret)
        print(f'Epoch {epoch + 1:2d},  Loss: {loss.item():9.4f},  Regret: {regret * 100:7.4f}%')
    print(f'Total Elapsed Time: {elapsed:.2f} Sec.')
    return loss_log, loss_log_regret

### 5.1 Smart Predict-then-Optimize (SPO+)

In [ ]:
spop = pyepo.func.SPOPlus(mpax_optmodel)
reg = LinearRegression()
if torch.cuda.is_available():
    reg = reg.cuda()
_, regret_spo = trainModel(reg, loss_func=spop, method_name='spo+')

### 5.2 Perturbed Fenchel-Young Loss (PFYL)

In [ ]:
pfyl = pyepo.func.perturbedFenchelYoung(mpax_optmodel, n_samples=3, sigma=1.0)
reg = LinearRegression()
if torch.cuda.is_available():
    reg = reg.cuda()
_, regret_pfy = trainModel(reg, loss_func=pfyl, method_name='pfy')

### 5.3 L2 Regularized Frank-Wolfe (RFWO + RFYL)

Frank-Wolfe needs only the linear minimization oracle that the MPAX model already provides, so it pairs naturally with `mpax_optmodel`. RFWO returns a regularized solution (use MSE as the task loss); RFYL returns a loss directly.

In [ ]:
rfwo = pyepo.func.regularizedFrankWolfeOpt(mpax_optmodel, lambd=1.0, max_iter=20, tol=1e-6)
reg = LinearRegression()
if torch.cuda.is_available():
    reg = reg.cuda()
criterion = nn.MSELoss()
_, regret_rfwo = trainModel(reg, loss_func=lambda cp, w: criterion(rfwo(cp), w), method_name='rfwo')

In [ ]:
rfyl = pyepo.func.regularizedFrankWolfeFenchelYoung(mpax_optmodel, lambd=1.0, max_iter=20, tol=1e-6)
reg = LinearRegression()
if torch.cuda.is_available():
    reg = reg.cuda()
_, regret_rfyl = trainModel(reg, loss_func=rfyl, method_name='rfyl')

### 5.4 Pointwise Learning to Rank

In [ ]:
ltr = pyepo.func.pointwiseLTR(mpax_optmodel, dataset=dataset_train, solve_ratio=0.5)
reg = LinearRegression()
if torch.cuda.is_available():
    reg = reg.cuda()
_, regret_ltr = trainModel(reg, loss_func=ltr, method_name='ltr')

## 6 Test Regret Comparison

Plot per-epoch test regret for all five methods. With an MPAX-backed model the per-step cost of every method is dominated by the JIT-compiled batched LP solve rather than Python overhead, so wall-clock differences track gradient-estimator cost more than solver cost.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
epochs = list(range(len(regret_spo)))
ax.plot(epochs, [r * 100 for r in regret_spo],  marker='o', label='SPO+')
ax.plot(epochs, [r * 100 for r in regret_pfy],  marker='s', label='PFYL')
ax.plot(epochs, [r * 100 for r in regret_rfwo], marker='^', label='RFWO + MSE')
ax.plot(epochs, [r * 100 for r in regret_rfyl], marker='v', label='RFYL')
ax.plot(epochs, [r * 100 for r in regret_ltr],  marker='D', label='Pointwise LTR')
ax.set_xlabel('Epoch'); ax.set_ylabel('Test regret (%)')
ax.set_title(f'Knapsack-LP (m={m}, k={k}): test regret on MPAX backend')
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

## 7 Takeaways

* **MPAX is for batched LPs.** A single `jax.vmap` call replaces a Python loop over individual `Gurobi.solve()` calls; `optDataset` picks this fast path automatically for MPAX-backed models. The Section 2 bar chart shows the headline speedup.
* **All `pyepo.func` loss modules work on MPAX models** -- the LMO contract is the same `setObj` / `solve` interface, so SPO+, PFYL, RFWO, RFYL, and LTR all compose without changes.
* **Use MPAX when** (i) the inner problem is a (relaxable) LP, (ii) you have many instances per gradient step (large batch or solve-heavy losses like SPO+), and (iii) you have a GPU. For tiny problems Gurobi's per-instance solve is so fast that the JIT and dispatch overhead of MPAX is not worth it.